In [2]:
import import_ipynb
from Recodificacion import *
from Codigos import *

In [3]:
import pandas as pd
import joblib

In [4]:
def features_histograma(path, prefix):
    import numpy as np

    f = {}

    hist, hist_norm = obtener_histograma_intensidades(path)

    # guardar todo
    f[f"{prefix}_hist"] = hist.tolist()
    f[f"{prefix}_hist_norm"] = hist_norm.tolist()
    f[f"{prefix}_hist_bins"] = list(range(256))

    # derivados
    hist_norm_safe = hist_norm[hist_norm > 0]

    f[f"{prefix}_hist_entropy"] = -np.sum(hist_norm_safe * np.log2(hist_norm_safe))
    f[f"{prefix}_hist_std"] = np.std(hist_norm)
    f[f"{prefix}_hist_mean"] = np.mean(hist_norm)
    f[f"{prefix}_hist_min"] = np.min(hist_norm)
    f[f"{prefix}_hist_max"] = np.max(hist_norm)

    mu = np.mean(hist_norm)
    f[f"{prefix}_hist_skew"] = np.mean((hist_norm - mu)**3)
    f[f"{prefix}_hist_kurt"] = np.mean((hist_norm - mu)**4)

    return f

In [5]:
def features_generales(path, prefix):
    f = {}

    # básicas
    f[f"{prefix}_rango"] = obtener_rango_dinamico(path)
    f[f"{prefix}_std"] = obtener_desv_estandar_brillo(path)

    # contraste
    f[f"{prefix}_michelson"] = obtener_contraste_michelson(path)

    # frecuencia
    f[f"{prefix}_hf"] = obtener_energia_altas_frecuencias(path)
    f[f"{prefix}_lf"] = obtener_energia_bajas_frecuencias(path)
    f[f"{prefix}_hlfr"] = obtener_hlfr(path)

    # bordes multi escala
    f[f"{prefix}_edges_s1"] = obtener_densidad_bordes(path, sigma=1.0)
    f[f"{prefix}_edges_s2"] = obtener_densidad_bordes(path, sigma=2.0)
    f[f"{prefix}_edges_s3"] = obtener_densidad_bordes(path, sigma=3.0)

    # gradiente
    f[f"{prefix}_grad_mean"] = obtener_magnitud_media_gradiente(path)
    f[f"{prefix}_grad_std"] = obtener_desv_estandar_gradiente(path)

    # textura
    f[f"{prefix}_glcm_contrast"] = obtener_contraste_textura(path)
    f[f"{prefix}_glcm_hom"] = obtener_homogeneidad_textura(path)
    f[f"{prefix}_glcm_energy"] = obtener_energia_textura(path)
    f[f"{prefix}_glcm_corr"] = obtener_correlacion_glcm(path)

    # color
    f[f"{prefix}_sat_mean"] = obtener_media_saturacion(path)
    f[f"{prefix}_sat_var"] = obtener_varianza_saturacion(path)

    return f

In [6]:
def extraer_features(path, prefix):
    f = {}

    f.update(features_generales(path, prefix))
    f.update(features_histograma(path, prefix))

    return f

In [7]:
def aplicar_transformaciones_log(df):
    """
    Replica las transformaciones realizadas durante entrenamiento.
    """

    if "orig_hf" in df.columns:
        df["orig_hf_log"] = np.log1p(df["orig_hf"])
        df.drop(columns=["orig_hf"], inplace=True)

    if "orig_lf" in df.columns:
        df["orig_lf_log"] = np.log1p(df["orig_lf"])
        df.drop(columns=["orig_lf"], inplace=True)

    if "orig_glcm_contrast" in df.columns:
        df["orig_glcm_contrast_log"] = np.log1p(
            df["orig_glcm_contrast"]
        )
        df.drop(columns=["orig_glcm_contrast"], inplace=True)

    if "orig_glcm_energy" in df.columns:
        df["orig_glcm_energy_log"] = np.log1p(
            df["orig_glcm_energy"]
        )
        df.drop(columns=["orig_glcm_energy"], inplace=True)

    return df

In [8]:
def eliminar_columnas_descartadas(df):

    cols_drop = [
        "orig_hist",
        "orig_hist_norm",
        "orig_hist_bins",
        "orig_hist_mean",
        "orig_hist_skew",
        "orig_hist_kurt",
    ]

    cols_drop = [c for c in cols_drop if c in df.columns]

    return df.drop(columns=cols_drop)

In [9]:
def aplicar_encoding(df):

    df["metodo_imagemagick"] = (
        df["metodo"] == "imagemagick"
    )

    df["metodo_pillow"] = (
        df["metodo"] == "pillow"
    )

    df["formato_entrada_png"] = (
        df["formato_entrada"] == "png"
    )

    df["formato_entrada_webp"] = (
        df["formato_entrada"] == "webp"
    )

    df["formato_salida_png"] = (
        df["formato_salida"] == "png"
    )

    df["formato_salida_webp"] = (
        df["formato_salida"] == "webp"
    )

    df = df.drop(
        columns=[
            "metodo",
            "formato_entrada",
            "formato_salida"
        ]
    )

    return df

In [10]:
def convertir_bool_a_int(df):

    cols_bool = df.select_dtypes(
        include="bool"
    ).columns

    df[cols_bool] = df[cols_bool].astype(int)

    return df

In [11]:
def aplicar_scaler(df, scaler):

    cols_escalar = list(
        scaler.feature_names_in_
    )

    df[cols_escalar] = scaler.transform(
        df[cols_escalar]
    )

    return df

In [12]:
FEATURES_FINALES = [
    'calidad',
    'orig_rango',
    'orig_std',
    'orig_michelson',
    'orig_hlfr',
    'orig_edges_s1',
    'orig_edges_s2',
    'orig_edges_s3',
    'orig_grad_mean',
    'orig_grad_std',
    'orig_glcm_hom',
    'orig_glcm_corr',
    'orig_sat_mean',
    'orig_sat_var',
    'orig_hist_entropy',
    'orig_hf_log',
    'orig_lf_log',
    'orig_glcm_contrast_log',
    'orig_glcm_energy_log',
    'metodo_imagemagick',
    'metodo_pillow',
    'formato_entrada_png',
    'formato_entrada_webp',
    'formato_salida_png',
    'formato_salida_webp'
]

In [13]:
import os
def preparar_features_modelo(
    img_path,
    metodo,
    formato_salida,
    calidad,
    scaler
):
    formato_entrada = os.path.splitext(img_path)[1].lower().replace(".", "")
    feats = extraer_features(
        img_path,
        prefix="orig"
    )

    fila = {
        **feats,
        "metodo": metodo,
        "formato_entrada": formato_entrada,
        "formato_salida": formato_salida,
        "calidad": calidad
    }

    X = pd.DataFrame([fila])

    X = eliminar_columnas_descartadas(X)

    X = aplicar_transformaciones_log(X)

    X = aplicar_encoding(X)

    X = convertir_bool_a_int(X)

    # asegurar columnas exactas
    for col in FEATURES_FINALES:
        if col not in X.columns:
            X[col] = 0

    X = X[FEATURES_FINALES]

    X = aplicar_scaler(X, scaler)

    return X

In [14]:
def predecir_calidad(
    img_path,
    metodo,
    formato_salida,
    calidad,
    modelo_path,
    scaler_path
):

    modelo = joblib.load(modelo_path)
    scaler = joblib.load(scaler_path)

    X = preparar_features_modelo(
        img_path,
        metodo,
        formato_salida,
        calidad,
        scaler
    )

    pred = modelo.predict(X)[0]

    return float(pred)

In [17]:
score = predecir_calidad(
    img_path="prueba_img/image.webp",
    metodo="ffmpeg",
    formato_salida="jpg",
    calidad=10,
    modelo_path="modelos/xgb_final.pkl",
    scaler_path="modelos/scaler.pkl"
)

print(score)

0.9875339865684509
